# Warehouse Inventory EDA

Explores the warehouse inventory: building sizes, where warehouses cluster, and when they were built.

### Libraries

In [ ]:
import json
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from shapely import STRtree
from shapely.geometry import box

### Settings

In [ ]:
DATA = Path("/Volumes/01-2TB/NJ-Truck-and-Warhouse-Pollution/data")  # external SSD
RAW = DATA / "raw"
INTERIM = DATA / "interim"
UTM = "EPSG:26918"            # UTM zone 18N (NAD83), metres

SQFT_PER_M2 = 10.7639
SIZE_CLASSES = [50_000, 100_000, 250_000, 500_000, 1_000_000, np.inf]   # sq ft
SIZE_LABELS = ["50-100k", "100-250k", "250-500k", "500k-1M", "1M+"]
PRECISION_CLASSES = [50_000, 100_000, 200_000, np.inf]                  # coarser: only 100 labels
PRECISION_LABELS = ["50-100k", "100-200k", "200k+"]
TURNPIKE_ROUTES = ["00000095__", "00000095W_"]   # HPMS route_id: I-95 is the Turnpike in this corridor, plus the western spur
TURNPIKE_KM = [1, 2, 5, 10]   # distance bands for the Turnpike comparison
GRID_KM = 2                   # cell size for the roof-area density map

# Chart style: blue for the main series, orange for a second one, grey ink for text and reference lines
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK_2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
BLUES = ["#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#184f95", "#0d366b"]
axis = dict(gridcolor=GRID, linecolor=INK_2, zeroline=False, ticks="outside", tickcolor=GRID)
pio.templates["eda"] = go.layout.Template(layout=dict(
    font=dict(family="Inter, Helvetica, Arial, sans-serif", size=12, color=INK_2),
    title=dict(x=0, xanchor="left", font=dict(size=15, color=INK)),
    paper_bgcolor=SURFACE, plot_bgcolor=SURFACE, colorway=[BLUE, ORANGE],
    xaxis=axis, yaxis=axis, hoverlabel=dict(bgcolor="white", font_color=INK),
    legend=dict(orientation="h", yanchor="bottom", y=1.0, xanchor="left", x=0),
    margin=dict(l=60, r=30, t=80, b=50),
))
pio.templates.default = "plotly_white+eda"
pio.renderers.default = "plotly_mimetype"   # renders in VS Code and JupyterLab

### Functions

In [ ]:
def wilson(k, n, z=1.96):
    """
    - returns (low, high) 95% Wilson interval for k of n
    """
    p = k / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centre - half, centre + half


def precision_row(g):
    """
    - returns labelled, warehouses, precision, 95% interval and area-weighted precision for given label group
    """
    k, n = g["is_warehouse"].sum(), len(g)
    lo, hi = wilson(k, n)
    return pd.Series({"labelled": n, "warehouses": k, "precision": k / n, "ci_low": lo, "ci_high": hi,
                      "area_weighted": g.loc[g["is_warehouse"], "area_sqft"].sum() / g["area_sqft"].sum()})


def land_share(km):
    """
    - returns share of study area within km of the Turnpike
    """
    return turnpike_line.buffer(km * 1000).intersection(study_area).area / study_area.area


def line_coords(geom):
    """
    - returns lon, lat lists for given (multi)line, None between parts
    """
    lons, lats = [], []
    for part in getattr(geom, "geoms", [geom]):
        xs, ys = part.xy
        lons += list(xs) + [None]
        lats += list(ys) + [None]
    return lons, lats


def add_context(fig):
    """
    - adds county boundaries and Turnpike lines to given map figure
    """
    fig.add_trace(go.Scattermap(lon=county_ll[0], lat=county_ll[1], mode="lines", line=dict(color="#9a998f", width=1),
                                name="County boundary", hoverinfo="skip"))
    fig.add_trace(go.Scattermap(lon=turnpike_ll[0], lat=turnpike_ll[1], mode="lines", line=dict(color=INK, width=2.5),
                                name="NJ Turnpike", hoverinfo="skip"))

### Load inventory

In [2]:
assert INTERIM.exists(), f"{INTERIM} not found; plug in the external SSD"

inventory = gpd.read_file(INTERIM / "warehouse_inventory.gpkg", layer="inventory").to_crs(UTM)
inventory["area_sqft"] = inventory["area_m2"] * SQFT_PER_M2
inventory["size_class"] = pd.cut(inventory["area_sqft"], SIZE_CLASSES, labels=SIZE_LABELS, right=False)
centres = inventory.centroid
lonlat = centres.to_crs("EPSG:4326")
inventory["lon"], inventory["lat"] = lonlat.x, lonlat.y

counties = gpd.read_file(RAW / "core_counties.geojson", columns=["NAME", "ALAND"]).to_crs(UTM)
study_area = counties.union_all()

print(f"{len(inventory):,} buildings, {inventory['area_sqft'].sum() / 1e6:,.0f} million sq ft of roof "
      f"({inventory['area_m2'].sum() / 1e6:.1f} km²)")

1,291 buildings, 191 million sq ft of roof (17.7 km²)


## 1. Size distribution

In [3]:
print(inventory["area_sqft"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).map("{:,.0f}".format).to_string())

by_size = inventory.groupby("size_class", observed=False).agg(buildings=("cand_id", "size"), roof_msqft=("area_sqft", "sum"))
by_size["roof_msqft"] /= 1e6
by_size["share_buildings"] = by_size["buildings"] / by_size["buildings"].sum()
by_size["share_roof"] = by_size["roof_msqft"] / by_size["roof_msqft"].sum()

top10 = inventory["area_sqft"].nlargest(int(np.ceil(len(inventory) * 0.1))).sum() / inventory["area_sqft"].sum()
print(f"\nThe largest 10% of buildings hold {top10:.0%} of the roof area")
by_size.style.format({"roof_msqft": "{:,.1f}", "share_buildings": "{:.0%}", "share_roof": "{:.0%}"})

count        1,291
mean       147,984
std        143,879
min         50,024
10%         55,515
25%         67,090
50%         99,116
75%        161,414
90%        299,866
99%        789,520
max      1,264,129

The largest 10% of buildings hold 34% of the roof area


,buildings,roof_msqft,share_buildings,share_roof
size_class,,,,
50-100k,655,45.8,51%,24%
100-250k,475,72.2,37%,38%
250-500k,114,39.3,9%,21%
500k-1M,43,29.3,3%,15%
1M+,4,4.4,0%,2%


Number of buildings by footprint size.

In [ ]:
edges = np.logspace(np.log10(50_000), np.log10(inventory["area_sqft"].max() * 1.01), 31)
counts, _ = np.histogram(inventory["area_sqft"], bins=edges)
bin_mid = np.sqrt(edges[:-1] * edges[1:])
bin_range = np.column_stack([edges[:-1], edges[1:]])
median = inventory["area_sqft"].median()

In [4]:
fig = go.Figure(go.Bar(
    x=bin_mid, y=counts, width=np.diff(edges) * 0.92, marker_color=BLUE, customdata=bin_range,
    hovertemplate="%{customdata[0]:,.0f}–%{customdata[1]:,.0f} sq ft<br><b>%{y} buildings</b><extra></extra>"))
fig.add_vline(x=median, line=dict(color=INK_2, dash="dash", width=1),
              annotation=dict(text=f"median {median / 1e3:,.0f}k sq ft", font_color=INK_2), annotation_position="top right")
fig.update_xaxes(type="log", title="Footprint area (sq ft, log scale)", showgrid=False,
                 tickvals=[50_000, 100_000, 250_000, 500_000, 1_000_000, 2_000_000],
                 ticktext=["50k", "100k", "250k", "500k", "1M", "2M"])
fig.update_yaxes(title="Buildings")
fig.update_layout(title=f"Warehouse footprint size, {len(inventory):,} buildings", height=420, bargap=0)
fig.show()

Share of buildings vs share of roof area, by size class.

In [ ]:
share_series = [("share_buildings", "Share of buildings", BLUE), ("share_roof", "Share of roof area", ORANGE)]
size_names = [f"{s} sq ft" for s in by_size.index]
share_text = {col: [f"{v:.0%}" for v in by_size[col]] for col, _, _ in share_series}
share_max = by_size[["share_buildings", "share_roof"]].max().max()

In [5]:
fig = go.Figure()
for col, name, color in share_series:
    fig.add_bar(y=size_names, x=by_size[col], name=name, orientation="h", marker_color=color,
                text=share_text[col], textposition="outside", textfont_color=INK_2,
                customdata=by_size[["buildings", "roof_msqft"]].to_numpy(),
                hovertemplate="%{y}<br>%{x:.0%} · %{customdata[0]:,} buildings · %{customdata[1]:.1f}M sq ft<extra>" + name + "</extra>")
fig.update_xaxes(tickformat=".0%", title=None, range=[0, share_max * 1.15])
fig.update_yaxes(autorange="reversed", showgrid=False)
fig.update_layout(title="Small buildings are most of the count; big ones hold much of the roof",
                  barmode="group", bargap=0.25, bargroupgap=0.08, height=380)
fig.show()

### Precision by size

In [6]:
export = sorted((INTERIM / "labels").glob("project-3-*.json"))[-1]   # newest precision export
rows = []
for task in json.load(open(export)):
    results = [r for a in task["annotations"] if not a["was_cancelled"] for r in a["result"] if r["type"] == "choices"]
    choices = {r["from_name"]: r["value"]["choices"] for r in results}
    rows.append({"cand_id": Path(task["data"]["image"]).stem,
                 "verdict": (choices.get("verdict") or [None])[0],
                 "unusable": "unusable" in choices.get("flags", [])})
labels = pd.DataFrame(rows).merge(inventory[["cand_id", "area_sqft"]], on="cand_id")
labels = labels[~labels["unusable"] & labels["verdict"].isin(["warehouse", "not_warehouse"])]
labels["is_warehouse"] = labels["verdict"] == "warehouse"
labels["size_class"] = pd.cut(labels["area_sqft"], PRECISION_CLASSES, labels=PRECISION_LABELS, right=False)
print(f"{export.name}: {len(labels)} usable labels")

precision_by_size = pd.DataFrame({c: precision_row(g) for c, g in labels.groupby("size_class", observed=False)}).T
precision_by_size.loc["all sizes"] = precision_row(labels)
precision_by_size.style.format({"labelled": "{:.0f}", "warehouses": "{:.0f}", "precision": "{:.0%}",
                                "ci_low": "{:.0%}", "ci_high": "{:.0%}", "area_weighted": "{:.0%}"})

project-3-at-2026-10-24-22-42-4f7e54dd.json: 99 usable labels


,labelled,warehouses,precision,ci_low,ci_high,area_weighted
50-100k,60,40,67%,54%,77%,69%
100-200k,26,21,81%,62%,91%,80%
200k+,13,12,92%,67%,99%,93%
all sizes,99,73,74%,64%,81%,80%


Share of labelled buildings that are warehouses, by size class.

In [ ]:
p = precision_by_size
precision_names = [f"{s} sq ft" if s != "all sizes" else s for s in p.index]
precision_text = [f"{v:.0%} (n={n:.0f})" for v, n in zip(p["precision"], p["labelled"])]

In [7]:
fig = go.Figure(go.Scatter(
    x=p["precision"], y=precision_names, mode="markers+text", marker=dict(color=BLUE, size=11, line=dict(color=SURFACE, width=2)),
    error_x=dict(type="data", symmetric=False, array=p["ci_high"] - p["precision"],
                 arrayminus=p["precision"] - p["ci_low"], color="#86b6ef", thickness=3, width=0),
    text=precision_text, textposition="top center", textfont=dict(color=INK_2, size=11),
    customdata=p[["warehouses", "labelled", "ci_low", "ci_high", "area_weighted"]].to_numpy(),
    hovertemplate=("%{y}<br><b>%{x:.0%}</b> (%{customdata[0]:.0f} of %{customdata[1]:.0f} are warehouses)"
                   "<br>95% interval %{customdata[2]:.0%}–%{customdata[3]:.0%}"
                   "<br>area-weighted %{customdata[4]:.0%}<extra></extra>")))
fig.update_xaxes(tickformat=".0%", range=[0.3, 1.02], title="Share of labelled buildings that are warehouses (95% interval)")
fig.update_yaxes(autorange="reversed", showgrid=False)
fig.update_layout(title="Inventory precision by building size", height=340)
fig.show()

## 2. Clustering
### By county and town

In [8]:
land_km2 = counties.set_index("NAME")["ALAND"] / 1e6
by_county = inventory.groupby("county").agg(buildings=("cand_id", "size"), roof_msqft=("area_sqft", "sum"),
                                            median_sqft=("area_sqft", "median"))
by_county["roof_msqft"] /= 1e6
by_county["share_roof"] = by_county["roof_msqft"] / by_county["roof_msqft"].sum()
by_county["buildings_per_100km2"] = by_county["buildings"] / land_km2.reindex(by_county.index) * 100
by_county.sort_values("roof_msqft", ascending=False).style.format(
    {"roof_msqft": "{:,.1f}", "median_sqft": "{:,.0f}", "share_roof": "{:.0%}", "buildings_per_100km2": "{:.1f}"}, na_rep="")

,buildings,roof_msqft,median_sqft,share_roof,buildings_per_100km2
county,,,,,
Middlesex,834,141.7,"111,393",74%,104.1
Union,243,28.1,"85,806",15%,91.3
Essex,214,21.3,"72,154",11%,65.5


Warehouse roof area for the top 15 towns.

In [ ]:
by_town = (inventory.groupby(["mun_name", "county"]).agg(buildings=("cand_id", "size"), roof_msqft=("area_sqft", "sum"))
           .sort_values("roof_msqft", ascending=False))
by_town["roof_msqft"] /= 1e6
by_town["cum_share_roof"] = by_town["roof_msqft"].cumsum() / by_town["roof_msqft"].sum()
print(f"{len(by_town)} towns have at least one inventory building; "
      f"the top 10 hold {by_town['cum_share_roof'].iloc[9]:.0%} of the roof area")

top = by_town.head(15).reset_index()
top["label"] = top["mun_name"].str.title() + " (" + top["county"] + ")"
top["text"] = top["roof_msqft"].map("{:.1f}M".format)

In [9]:
fig = go.Figure(go.Bar(
    y=top["label"], x=top["roof_msqft"], orientation="h", marker_color=BLUE,
    text=top["text"], textposition="outside", textfont_color=INK_2,
    customdata=top[["buildings", "cum_share_roof"]].to_numpy(),
    hovertemplate="%{y}<br><b>%{x:.1f}M sq ft</b> · %{customdata[0]} buildings"
                  "<br>top towns so far: %{customdata[1]:.0%} of all roof area<extra></extra>"))
fig.update_xaxes(title="Warehouse roof area (million sq ft)", range=[0, top["roof_msqft"].max() * 1.15])
fig.update_yaxes(autorange="reversed", showgrid=False)
fig.update_layout(title="Top 15 towns by warehouse roof area", height=520)
fig.show()

52 towns have at least one inventory building; the top 10 hold 73% of the roof area


### NJ Turnpike

In [10]:
hpms = gpd.read_file(RAW / "traffic" / "hpms_core.gpkg", layer="hpms_2024").to_crs(UTM)
turnpike = hpms[hpms["route_id"].isin(TURNPIKE_ROUTES)]
assert not turnpike.empty, "no Turnpike segments found; check TURNPIKE_ROUTES against hpms['route_id']"
print(f"{len(turnpike):,} Turnpike segments, {turnpike.length.sum() / 1000:.0f} km of carriageway")

turnpike_line = turnpike.union_all()
inventory["turnpike_km"] = centres.distance(turnpike_line) / 1000

579 Turnpike segments, 67 km of carriageway


In [11]:
rows = []
for km in TURNPIKE_KM:
    near = inventory["turnpike_km"] <= km
    roof = inventory.loc[near, "area_sqft"].sum() / inventory["area_sqft"].sum()
    rows.append({"within_km": km, "buildings": near.sum(), "share_buildings": near.mean(),
                 "share_roof": roof, "share_land": land_share(km), "concentration": roof / land_share(km)})
turnpike_bands = pd.DataFrame(rows).set_index("within_km")
print(f"Median distance to the Turnpike: {inventory['turnpike_km'].median():.1f} km")
turnpike_bands.style.format({"share_buildings": "{:.0%}", "share_roof": "{:.0%}", "share_land": "{:.0%}",
                             "concentration": "{:.1f}×"})

Median distance to the Turnpike: 3.0 km


,buildings,share_buildings,share_roof,share_land,concentration
within_km,,,,,
1,248,19%,25%,9%,2.9×
2,516,40%,50%,17%,3.0×
5,803,62%,71%,38%,1.9×
10,1115,86%,91%,70%,1.3×


Share of warehouse roof area vs share of land within each distance of the Turnpike.

In [ ]:
d = np.round(np.arange(0, inventory["turnpike_km"].max() + 0.5, 0.5), 1)
order = inventory.sort_values("turnpike_km")
roof_cum = np.interp(d, order["turnpike_km"], order["area_sqft"].cumsum() / order["area_sqft"].sum(), left=0)
land_cum = np.array([land_share(km) for km in d])

In [12]:
fig = go.Figure()
fig.add_scatter(x=d, y=roof_cum, name="Warehouse roof area", line=dict(color=BLUE, width=2.5))
fig.add_scatter(x=d, y=land_cum, name="Land area", line=dict(color=INK_2, width=1.5, dash="dash"))
fig.update_traces(hovertemplate="%{y:.0%}")
fig.update_xaxes(title="Distance from the NJ Turnpike (km)", range=[0, d.max()])
fig.update_yaxes(title="Share within that distance", tickformat=".0%", range=[0, 1.02])
fig.update_layout(title="Warehouse roof area sits closer to the Turnpike than the land does",
                  hovermode="x unified", height=420)
fig.show()

### Maps
Inventory buildings, sized by footprint.

In [ ]:
turnpike_ll = line_coords(gpd.GeoSeries([turnpike_line], crs=UTM).to_crs("EPSG:4326").iloc[0])
county_ll = line_coords(counties.to_crs("EPSG:4326").boundary.union_all())
map_view = dict(style="carto-positron", center=dict(lon=inventory["lon"].mean(), lat=inventory["lat"].mean()), zoom=9.2)

hover = inventory.assign(town=inventory["mun_name"].str.title(),
                         built=inventory["year_built"].map(lambda y: "unknown" if pd.isna(y) else f"{y:.0f}"))

In [13]:
fig = go.Figure()
add_context(fig)
fig.add_trace(go.Scattermap(
    lon=hover["lon"], lat=hover["lat"], mode="markers", name="Inventory building",
    marker=dict(size=np.sqrt(hover["area_sqft"]) / 45, sizemin=3, color=BLUE, opacity=0.65),
    customdata=hover[["cand_id", "town", "county", "area_sqft", "built", "turnpike_km"]].to_numpy(),
    hovertemplate=("<b>%{customdata[0]}</b> · %{customdata[1]}, %{customdata[2]}"
                   "<br>%{customdata[3]:,.0f} sq ft · built %{customdata[4]}"
                   "<br>%{customdata[5]:.1f} km from the Turnpike<extra></extra>")))
fig.update_layout(title="Inventory buildings, sized by footprint", map=map_view, height=700,
                  margin=dict(l=0, r=0, t=80, b=0))
fig.show()

Warehouse roof area per 2 km grid cell.

In [ ]:
minx, miny, maxx, maxy = study_area.bounds
step = GRID_KM * 1000
xs = np.arange(minx // step * step, maxx, step)
ys = np.arange(miny // step * step, maxy, step)
grid = gpd.GeoDataFrame(geometry=[box(x, y, x + step, y + step) for x in xs for y in ys], crs=UTM)
grid = grid[grid.intersects(study_area)].reset_index(drop=True)

in_cell = gpd.sjoin(gpd.GeoDataFrame(inventory[["area_sqft"]], geometry=centres, crs=UTM), grid, predicate="within")
cells = in_cell.groupby("index_right").agg(buildings=("area_sqft", "size"), roof_msqft=("area_sqft", "sum"))
grid = grid.join(cells, how="inner")
grid["roof_msqft"] /= 1e6
grid["cell_id"] = grid.index.astype(str)
grid_geojson = json.loads(grid.to_crs("EPSG:4326").to_json())
grid_colorscale = [[i / (len(BLUES) - 1), c] for i, c in enumerate(BLUES)]
print(f"{len(grid)} of the {GRID_KM} km cells have a warehouse; the busiest 10% hold "
      f"{grid['roof_msqft'].nlargest(int(np.ceil(len(grid) * 0.1))).sum() / grid['roof_msqft'].sum():.0%} of the roof area")

In [14]:
fig = go.Figure(go.Choroplethmap(
    geojson=grid_geojson, featureidkey="properties.cell_id",
    locations=grid["cell_id"], z=grid["roof_msqft"], colorscale=grid_colorscale,
    marker=dict(opacity=0.8, line=dict(color="white", width=0.5)), name="Grid cell",
    colorbar=dict(title=dict(text="Roof area<br>(million sq ft)", font_size=11), thickness=12, len=0.5),
    customdata=grid[["buildings"]].to_numpy(),
    hovertemplate="<b>%{z:.2f}M sq ft</b> of warehouse roof<br>%{customdata[0]} buildings<extra></extra>"))
add_context(fig)
fig.update_layout(title=f"Warehouse roof area per {GRID_KM} km cell", map=map_view, height=700,
                  margin=dict(l=0, r=0, t=80, b=0), legend=dict(y=1.0))
fig.show()

178 of the 2 km cells have a warehouse; the busiest 10% hold 46% of the roof area


### Nearest-neighbour clustering

In [15]:
pts = np.array(centres.values)
_, nn_dist = STRtree(pts).query_nearest(pts, exclusive=True, return_distance=True, all_matches=False)
observed = nn_dist.mean()
expected = 0.5 / np.sqrt(len(pts) / study_area.area)
print(f"Mean nearest-neighbour distance: {observed:,.0f} m (random would be {expected:,.0f} m)")
print(f"Clark-Evans ratio R = {observed / expected:.2f}")
print(f"Median nearest neighbour: {np.median(nn_dist):,.0f} m; "
      f"{(nn_dist <= 250).mean():.0%} of warehouses have another within 250 m")

Mean nearest-neighbour distance: 248 m (random would be 529 m)
Clark-Evans ratio R = 0.47
Median nearest neighbour: 179 m; 76% of warehouses have another within 250 m


## 3. Year built (MOD-IV YR_CONSTR)

In [16]:
inventory["has_year"] = inventory["year_built"].notna()
print(f"Usable year built: {inventory['has_year'].sum():,} of {len(inventory):,} ({inventory['has_year'].mean():.0%})")
coverage = pd.concat({
    "county": inventory.groupby("county")["has_year"].agg(["size", "mean"]),
    "size class": inventory.groupby("size_class", observed=False)["has_year"].agg(["size", "mean"]),
}).rename(columns={"size": "buildings", "mean": "share_with_year"})
coverage.style.format({"share_with_year": "{:.0%}"})

Usable year built: 529 of 1,291 (41%)


Buildings and roof area by 5-year period built.

In [ ]:
dated = inventory[inventory["has_year"]].copy()
dated["period"] = (dated["year_built"] // 5 * 5).astype(int)
by_period = dated.groupby("period").agg(buildings=("cand_id", "size"), roof_msqft=("area_sqft", "sum"))
by_period["roof_msqft"] /= 1e6
by_period = by_period.reindex(range(by_period.index.min(), by_period.index.max() + 1, 5), fill_value=0)
period_label = [f"{p}–{p + 4}" for p in by_period.index]
period_rows = [(1, "buildings", "buildings"), (2, "roof_msqft", "M sq ft")]

In [17]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08)
for row, col, unit in period_rows:
    fig.add_bar(x=by_period.index + 2.5, y=by_period[col], width=4.4, marker_color=BLUE, row=row, col=1, showlegend=False,
                customdata=period_label, hovertemplate="%{customdata}<br><b>%{y:,.1f} " + unit + "</b><extra></extra>")
fig.update_yaxes(title="Buildings", row=1, col=1)
fig.update_yaxes(title="Roof area (M sq ft)", row=2, col=1)
fig.update_xaxes(showgrid=False)
fig.update_xaxes(title="Year built (MOD-IV YR_CONSTR), 5-year periods", row=2, col=1)
fig.update_layout(title=f"Year built ({len(dated):,} of {len(inventory):,} buildings have a year)", height=560, bargap=0)
fig.show()

In [18]:
decades = dated.assign(decade=(dated["year_built"] // 10 * 10).astype(int))
decade_table = decades.groupby("decade").agg(buildings=("cand_id", "size"), median_sqft=("area_sqft", "median"),
                                             roof_msqft=("area_sqft", "sum"))
decade_table["roof_msqft"] /= 1e6
decade_table["cum_share_roof"] = decade_table["roof_msqft"].cumsum() / decade_table["roof_msqft"].sum()
decade_table.style.format({"median_sqft": "{:,.0f}", "roof_msqft": "{:,.1f}", "cum_share_roof": "{:.0%}"})

,buildings,median_sqft,roof_msqft,cum_share_roof
decade,,,,
1890,2,"67,519",0.1,0%
1900,8,"83,421",1.1,2%
1910,4,"84,898",0.3,2%
1920,14,"88,131",1.9,5%
1930,13,"98,285",1.7,7%
1940,17,"91,549",1.8,9%
1950,34,"95,415",4.6,16%
1960,79,"81,984",8.2,27%
1970,138,"88,676",16.9,49%
